# 실습 9. 로지스틱 회귀분석

사회약학 연구방법 노트의 '실습 9. 로지스틱 회귀분석'에 딸린 노트북입니다. 앞부분(가–다)은 MASS 패키지의 birthwt 자료(출생아와 산모 189명)로 저체중아 출산과 산모 특성의 관계를 다변수 로지스틱 회귀로 분석하고 모형을 점검합니다. 뒷부분(라)은 scikit-learn의 위스콘신 진단 유방암 자료로 예측모형을 만들고 판별과 교정을 평가합니다. 셀을 위에서부터 차례로 실행하세요. 설명과 출력 읽는 법은 사이트의 실습 9에 있습니다.


## 가. 실습 데이터 준비

이 실습이 쓰는 패키지를 불러와 버전을 확인한 뒤 birthwt 자료를 읽습니다. 자료는 인터넷에서 읽어 오므로 인터넷 연결이 필요합니다.

### 셀 1. 패키지 불러오기와 버전 확인

In [ ]:
import pandas as pd
import numpy as np
import statsmodels, scipy, matplotlib, sklearn

print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("statsmodels ", statsmodels.__version__)
print("scipy       ", scipy.__version__)
print("matplotlib  ", matplotlib.__version__)
print("scikit-learn", sklearn.__version__)

### 셀 2. 출생체중 자료 불러오기

In [ ]:
url = "https://vincentarelbundock.github.io/Rdatasets/csv/MASS/birthwt.csv"
bw = pd.read_csv(url)        # 인터넷의 CSV 파일을 DataFrame으로 읽음
print(bw.shape)
bw.head()

### 셀 3. 결과변수와 설명변수 코딩

In [ ]:
print(bw["low"].value_counts())                   # 1 = 저체중아
print((bw["bwt"] < 2500).equals(bw["low"] == 1))  # 정의 확인
bw["race"] = pd.Categorical(
    bw["race"].map({1: "white", 2: "black", 3: "other"}),
    categories=["white", "black", "other"])       # 첫 범주가 기준
bw["ptd"] = (bw["ptl"] > 0).astype(int)   # 이전 조산 1회 이상 = 1
pd.crosstab(bw["ptl"], bw["low"], margins=True)

### 셀 4. 사건 수와 EPV

In [ ]:
n_event = int(bw["low"].sum())            # 사건(저체중아) 수
n_non = int((bw["low"] == 0).sum())       # 사건이 없는 수
k = 8     # 절편 외 계수: age, lwt, race 2개, smoke, ptd, ht, ui
print("events:", n_event, " non-events:", n_non)
print("proportion:", round(n_event / len(bw), 3))
print("EPV =", round(min(n_event, n_non) / k, 1))

## 나. 다변수 로지스틱 회귀

### 셀 5. 다변수 로지스틱 회귀 적합

In [ ]:
import statsmodels.formula.api as smf

f = "low ~ age + lwt + C(race) + smoke + ptd + ht + ui"
logit_m = smf.logit(f, data=bw).fit()     # 최대우도추정
print(logit_m.summary())

### 셀 6. 오즈비와 95% 신뢰구간 표

In [ ]:
ci = logit_m.conf_int()                  # 로그 오즈비의 95% CI
or_tab = pd.DataFrame({
    "OR": np.exp(logit_m.params),        # e^β = 오즈비
    "lower": np.exp(ci[0]),
    "upper": np.exp(ci[1]),
    "P": logit_m.pvalues})
or_tab.drop("Intercept").round(3)

### 셀 7. 임상적으로 의미 있는 단위의 오즈비

In [ ]:
def or_per(m, var, c):                  # c 단위당 오즈비와 95% CI
    est = m.params[var]                 # 1단위당 로그 오즈비
    lo, hi = m.conf_int().loc[var]      # 그 95% 신뢰구간
    return np.round(np.exp([c * est, c * lo, c * hi]), 3)

crude_lwt = smf.logit("low ~ lwt", data=bw).fit(disp=0)
crude_age = smf.logit("low ~ age", data=bw).fit(disp=0)
print("lwt per 10 lb, adjusted:", or_per(logit_m, "lwt", 10))
print("lwt per 10 lb, crude   :", or_per(crude_lwt, "lwt", 10))
print("age per 5 yr,  adjusted:", or_per(logit_m, "age", 5))
print("age per 5 yr,  crude   :", or_per(crude_age, "age", 5))

### 셀 8. 보정 전후 오즈비 비교

In [ ]:
def or_ci(m):
    ci = np.exp(m.conf_int())
    f2 = "{:.2f}".format
    return (np.exp(m.params).map(f2) + " (" + ci[0].map(f2)
            + " to " + ci[1].map(f2) + ")")

terms = ["age", "lwt", "C(race)", "smoke", "ptd", "ht", "ui"]
crude = pd.concat([or_ci(smf.logit(f"low ~ {t}", data=bw).fit(disp=0))
                   .drop("Intercept") for t in terms])
adj = or_ci(logit_m)[crude.index]         # 같은 순서로 맞춤
pd.DataFrame({"crude OR (95% CI)": crude,
              "adjusted OR (95% CI)": adj})

### 셀 9. 예시 산모의 예측 확률

In [ ]:
new = pd.DataFrame({
    "age":   [25, 25, 20, 20],
    "lwt":   [120, 120, 100, 100],
    "race":  ["white", "white", "black", "black"],
    "smoke": [0, 1, 0, 1],
    "ptd":   [0, 0, 1, 1],
    "ht":    [0, 0, 0, 0],
    "ui":    [0, 0, 1, 1]})
new["p_low"] = logit_m.predict(new).round(3)   # 예측 확률
new

### 셀 10. 흔한 결과에서 오즈비와 상대위험도

In [ ]:
tab = pd.crosstab(bw["smoke"], bw["low"])   # 행 smoke, 열 low
risk = tab[1] / tab.sum(axis=1)             # 저체중아 비율(위험)
odds = tab[1] / tab[0]                      # 오즈
print(tab)
print("risk:", risk.round(3).tolist())
print("RR =", round(risk[1] / risk[0], 2),
      "  OR =", round(odds[1] / odds[0], 2))

### 셀 11. 모형 기반 표준화로 보정 상대위험도 구하기

In [ ]:
r1 = logit_m.predict(bw.assign(smoke=1)).mean()   # 모두 흡연했다면
r0 = logit_m.predict(bw.assign(smoke=0)).mean()   # 모두 비흡연이라면
print(f"standardized risk: smokers {r1:.3f}, non-smokers {r0:.3f}")
print(f"adjusted RR = {r1 / r0:.2f}, RD = {r1 - r0:.3f}")

결과변수를 1/2로 넣어 일부러 오류를 내 보는 셀(셀 12)은 이 노트북에서 뺐습니다(사이트의 실습 9 나 절에 있습니다).

## 다. 모형 점검

### 셀 13. lwt의 로짓 선형성: 스플라인과 비교

In [ ]:
from scipy import stats

f_sp = "low ~ age + bs(lwt, df=4) + C(race) + smoke + ptd + ht + ui"
spline_m = smf.logit(f_sp, data=bw).fit(disp=0)
lr = 2 * (spline_m.llf - logit_m.llf)              # 우도비 통계량
ddf = spline_m.df_model - logit_m.df_model         # 늘어난 계수 수
p = stats.chi2.sf(lr, ddf)                         # p-value
print(f"LR = {lr:.2f}, df = {ddf:.0f}, P = {p:.3f}")
print("AIC linear:", round(logit_m.aic, 1),
      " spline:", round(spline_m.aic, 1))

### 셀 14. 직선 모형과 스플라인 모형의 로그 오즈

In [ ]:
import matplotlib.pyplot as plt

grid = pd.DataFrame({"lwt": np.arange(85, 251, 5)}).assign(
    age=23, race="white", smoke=0, ptd=0, ht=0, ui=0)
def log_odds(m):
    p = m.predict(grid)                 # 예측 확률
    return np.log(p / (1 - p))          # 확률 -> 로그 오즈
plt.figure(figsize=(6, 3.8))
plt.plot(grid["lwt"], log_odds(logit_m), label="Linear in lwt")
plt.plot(grid["lwt"], log_odds(spline_m), "--",
         label="B-spline (df = 4)")
plt.plot(bw["lwt"], np.full(len(bw), -8), "|", color="gray")  # 관측값
plt.xlabel("Mother's weight at last menstrual period (lb)")
plt.ylabel("Log odds of low birth weight")
plt.legend()
plt.show()

### 셀 15. 영향력이 큰 관측값 찾기

In [ ]:
import statsmodels.api as sm

glm_m = smf.glm(f, data=bw, family=sm.families.Binomial()).fit()
print("same coefficients:", np.allclose(glm_m.params, logit_m.params))
infl = glm_m.get_influence().summary_frame()     # 관측값별 영향력
cols = ["cooks_d", "standard_resid", "hat_diag"]
top = infl[cols].sort_values("cooks_d", ascending=False).head(5)
top.join(bw[["low", "age", "lwt", "race", "smoke", "ptd", "ht",
             "ui"]]).round(3)

### 셀 16. Cook의 거리 그림

In [ ]:
plt.figure(figsize=(8, 3))
plt.stem(infl.index, infl["cooks_d"], markerfmt=".", basefmt=" ")
plt.xlabel("Observation (row index)")
plt.ylabel("Cook's distance")
plt.title("Influence of each mother on the fitted model")
plt.show()

### 셀 17. 가장 영향력이 큰 1명을 빼고 다시 적합

In [ ]:
m_wo = smf.logit(f, data=bw.drop(index=top.index[0])).fit(disp=0)
comp = pd.DataFrame({"OR all (n=189)": np.exp(logit_m.params),
                     "OR without 1": np.exp(m_wo.params)})
comp.drop("Intercept").round(2)

### 셀 18. 분리가 생기는 모형: ptl을 범주 그대로 넣으면

In [ ]:
f_ptl = "low ~ age + lwt + C(race) + smoke + C(ptl) + ht + ui"
sep_m = smf.logit(f_ptl, data=bw).fit()
print(sep_m.summary().tables[1])        # 계수 표만 출력
print("converged:", sep_m.mle_retvals["converged"])

### 셀 19. 다중공선성: 분산팽창계수(VIF)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

Xd = logit_m.model.exog              # 설계행렬 (상수항 열 + 설명변수 8열)
names = logit_m.model.exog_names     # 각 열의 이름
vif = pd.Series([variance_inflation_factor(Xd, j)
                 for j in range(Xd.shape[1])], index=names, name="VIF")
vif.drop("Intercept").round(2)       # 상수항 행은 빼고 읽음

## 라. 예측모형과 성능 평가

유방암 자료는 scikit-learn에 들어 있어 인터넷 없이 불러옵니다. 그림과 마지막 셀에는 앞에서 불러온 matplotlib(셀 14)과 statsmodels(셀 15)를 그대로 씁니다.

### 셀 20. 유방암 자료 불러오기와 결과 코딩

In [ ]:
from sklearn.datasets import load_breast_cancer

bc = load_breast_cancer(as_frame=True)
X = bc.data                          # 설명변수 30개 (DataFrame)
print(X.shape)
print(bc.target_names)               # target 0, 1의 이름
print(bc.target.value_counts())
y = (bc.target == 0).astype(int)     # 악성 = 1, 양성 = 0 으로 바꿈
print("malignant (y = 1):", y.sum(), " proportion:", round(y.mean(), 3))

### 셀 21. 훈련 자료와 시험 자료로 나누기

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)
print(round(y_train.mean(), 3), round(y_test.mean(), 3))

### 셀 22. 표준화 + 로지스틱 회귀 모형 학습

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

clf = make_pipeline(StandardScaler(),
                    LogisticRegression(max_iter=1000))
clf.fit(X_train, y_train)            # 훈련 자료로만 학습
print(clf.classes_)                  # predict_proba 열의 순서

### 셀 23. predict_proba 결과 살펴보기

In [ ]:
proba = clf.predict_proba(X_test)    # 시험 자료 171명의 확률
print(type(proba), proba.shape)
print(proba[:5].round(3))            # 앞 5명, 두 열
p_test = proba[:, 1]                 # 모든 행, 1번 열 = P(y = 1)
print(p_test[:5].round(3))
print(proba[:5].sum(axis=1))         # 각 행의 합은 1

### 셀 24. ROC 곡선과 AUC

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

auc = roc_auc_score(y_test, p_test)          # (실제 0/1, 예측 확률)
fpr, tpr, thr = roc_curve(y_test, p_test)
print(f"AUC = {auc:.4f}")
plt.figure(figsize=(4.6, 4.4))
plt.plot(fpr, tpr, label=f"Logistic model (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Chance")
plt.xlabel("1 - Specificity (false positive rate)")
plt.ylabel("Sensitivity (true positive rate)")
plt.legend(loc="lower right")
plt.show()

### 셀 25. AUC의 부트스트랩 95% 신뢰구간

In [ ]:
rng = np.random.default_rng(2026)          # 난수 시드 고정
yt, aucs = y_test.to_numpy(), []
for _ in range(1000):
    i = rng.integers(0, len(yt), len(yt))  # 복원추출한 행 번호
    if yt[i].min() == yt[i].max():         # 한 범주만 뽑히면 건너뜀
        continue
    aucs.append(roc_auc_score(yt[i], p_test[i]))
print(len(aucs), np.percentile(aucs, [2.5, 97.5]).round(3))

### 셀 26. 혼동행렬

In [ ]:
from sklearn.metrics import confusion_matrix

pred05 = (p_test >= 0.5).astype(int)        # 0.5 이상이면 악성으로
print(confusion_matrix(y_test, pred05))      # 행 = 실제, 열 = 예측

### 셀 27. 임계값별 민감도, 특이도, 정확도

In [ ]:
def classify(t):
    pred = (p_test >= t).astype(int)         # 임계값 t 이상 = 악성
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    return {"threshold": t, "TP": tp, "FN": fn, "FP": fp, "TN": tn,
            "sensitivity": tp / (tp + fn),
            "specificity": tn / (tn + fp),
            "accuracy": (tp + tn) / len(pred)}

pd.DataFrame([classify(t) for t in [0.5, 0.2]]).round(3)

### 셀 28. 교정 곡선

In [ ]:
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(
    y_test, p_test, n_bins=5, strategy="quantile")
print("mean predicted:", prob_pred.round(3))
print("observed      :", prob_true.round(3))
plt.figure(figsize=(4.6, 4.4))
plt.plot(prob_pred, prob_true, "o-", label="Model (5 groups)")
plt.plot([0, 1], [0, 1], "--", color="gray",
         label="Perfect calibration")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed proportion malignant")
plt.legend()
plt.show()

### 셀 29. Brier 점수와 예측 확률의 분포

In [ ]:
from sklearn.metrics import brier_score_loss

brier = brier_score_loss(y_test, p_test)
p_ref = np.full(len(y_test), y_train.mean())   # 모두에게 같은 확률
print(f"Brier (model)    : {brier:.4f}")
print(f"Brier (reference): {brier_score_loss(y_test, p_ref):.4f}")
plt.figure(figsize=(6, 3))
plt.hist(p_test, bins=20)
plt.xlabel("Predicted probability of malignancy")
plt.ylabel("Number of patients")
plt.show()

### 셀 30. AUC와 정확도는 다른 것을 잽니다

In [ ]:
p_half = p_test / 2                        # 모든 확률을 절반으로
acc = lambda p: ((p >= 0.5).astype(int) == y_test).mean()
print("AUC     :", round(roc_auc_score(y_test, p_test), 4), "->",
      round(roc_auc_score(y_test, p_half), 4))
print("Brier   :", round(brier_score_loss(y_test, p_test), 4), "->",
      round(brier_score_loss(y_test, p_half), 4))
print("accuracy:", round(acc(p_test), 3), "->", round(acc(p_half), 3))
print("accuracy if all benign:", round((y_test == 0).mean(), 3))

### 셀 31. statsmodels와 scikit-learn의 계수 비교

In [ ]:
cols3 = ["mean radius", "mean texture", "mean smoothness"]
Z = StandardScaler().fit_transform(X_train[cols3])     # 표준화
sm_fit = sm.Logit(y_train.to_numpy(), sm.add_constant(Z)).fit(disp=0)
sk_c1 = LogisticRegression().fit(Z, y_train)            # 기본 C = 1
sk_big = LogisticRegression(C=1e6).fit(Z, y_train)      # 벌점 거의 없음
pd.DataFrame({"statsmodels": sm_fit.params[1:],
              "sklearn C=1": sk_c1.coef_[0],
              "sklearn C=1e6": sk_big.coef_[0]}, index=cols3).round(3)

## 과제 정답

**과제 1.** 셀 6의 오즈비 표에서 ht 행(고혈압 병력)을 한 문장으로 옮기고, 이 숫자가 말해 주지 않는 것을 두 가지 이상 쓰세요. 셀 11의 표준화 방법으로 고혈압 병력의 위험의 비를 구해 오즈비와 비교하세요.

### 셀 32. 과제 1 정답. 고혈압 병력의 오즈비 읽기

In [ ]:
print(pd.crosstab(bw["ht"], bw["low"], margins=True))
print("adjusted OR (95% CI):",
      or_tab.loc["ht", ["OR", "lower", "upper"]].round(2).tolist())
r_ht1 = logit_m.predict(bw.assign(ht=1)).mean()   # 모두 고혈압 병력이 있다면
r_ht0 = logit_m.predict(bw.assign(ht=0)).mean()   # 모두 없다면
print(f"standardized risk: ht=1 {r_ht1:.3f}, ht=0 {r_ht0:.3f}")
print(f"adjusted RR = {r_ht1 / r_ht0:.2f}")

**과제 2.** 인종의 기준범주를 white에서 black으로, 체중(lwt)의 단위를 1 lb에서 10 lb로 바꿔 셀 5의 모형을 다시 적합하세요. 어떤 오즈비가 바뀌고 어떤 오즈비가 그대로인지, 흡연의 오즈비는 왜 그대로인지 설명하세요.

### 셀 33. 과제 2 정답. 기준범주와 단위 바꾸기

In [ ]:
bw["race_b"] = pd.Categorical(bw["race"],
                              categories=["black", "white", "other"])
bw["lwt10"] = bw["lwt"] / 10                 # 체중을 10 lb 단위로
f_b = "low ~ age + lwt10 + C(race_b) + smoke + ptd + ht + ui"
m_b = smf.logit(f_b, data=bw).fit(disp=0)
print("log-likelihood:", round(logit_m.llf, 3), round(m_b.llf, 3))
print("same predictions:",
      np.allclose(logit_m.predict(bw), m_b.predict(bw)))
np.exp(m_b.params).round(3).rename("OR")

**과제 3.** 동료가 일곱 변수를 하나씩 넣은 단변수 로지스틱 회귀에서 P < 0.05인 변수만 골라 다변수 모형을 적합하고 "보정 후 흡연은 저체중아와 관련이 없었다(OR 1.65, 95% CI 0.83–3.26, P = 0.15)"라고 썼습니다. 동료의 분석을 재현하고, 이 결론에서 틀린 곳 두 가지를 찾아 왜 틀렸는지 설명하세요.

### 셀 34. 과제 3 정답. 단변수 P로 변수를 고른 분석

In [ ]:
keep = []
for t in terms:                                 # 셀 8의 일곱 변수
    m1 = smf.logit(f"low ~ {t}", data=bw).fit(disp=0)
    p1 = m1.pvalues.drop("Intercept").min()     # 가변수가 둘이면 작은 쪽
    print(f"{t:8} univariable P = {p1:.3f}")
    if p1 < 0.05:
        keep.append(t)

f_sel = "low ~ " + " + ".join(keep)            # 동료가 고른 모형
m_sel = smf.logit(f_sel, data=bw).fit(disp=0)
print(f_sel)
print("smoke, selected     :", or_per(m_sel, "smoke", 1),
      " P =", round(m_sel.pvalues["smoke"], 3))
print("smoke, prespecified :", or_per(logit_m, "smoke", 1))
bw.groupby("race", observed=True)[["smoke", "low"]].mean().round(3)  # 인종별 비율